In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from model import NSConfig, NsModel, CUR_DIR

In [2]:
from tokenizers import Tokenizer
tokenizer = Tokenizer.from_file(f"{CUR_DIR}/model/tokenizer_sft.json")

In [4]:
def apply_chat_template(user_prompt: str):
    system_prompt = "You are a helpful assistant."
    user_text = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_prompt}<|im_end|>\n"
    return user_text

In [3]:
cfg = NSConfig()

device = "cuda:1"

cfg.vocab_size += 2
model = NsModel(cfg)
model.to(device)

model.load_state_dict(
    torch.load(f"{CUR_DIR}/model/sft_model_step_700.pt", map_location=device, weights_only=True)
)
model.to(device)
model.eval()


NsModel(
  (emb): Embedding(32002, 768)
  (layers): ModuleList(
    (0-11): 12 x TransformerBlock(
      (rms_norm1): RMSNorm()
      (attn): GQAAttention(
        (w_q): Linear(in_features=768, out_features=768, bias=False)
        (w_k): Linear(in_features=768, out_features=128, bias=False)
        (w_v): Linear(in_features=768, out_features=128, bias=False)
        (w_o): Linear(in_features=768, out_features=768, bias=False)
      )
      (rms_norm2): RMSNorm()
      (ffn): FeedForward(
        (fc1): Linear(in_features=768, out_features=2048, bias=False)
        (fc2): Linear(in_features=768, out_features=2048, bias=False)
        (fc3): Linear(in_features=2048, out_features=768, bias=False)
      )
    )
  )
  (final_norm): RMSNorm()
  (lm_out): Linear(in_features=768, out_features=32002, bias=False)
)

In [9]:
# inference with attention mask

prompts = [
    apply_chat_template("tell me a joke"),
    apply_chat_template("write a python sort function")
]

pad_token_id = tokenizer.token_to_id("<pad>")

input_ids = tokenizer.encode_batch(prompts, add_special_tokens=False)

max_len = max(len(ids) for ids in input_ids)

input_list = []
mask_list = []

for item in input_ids:
    ids = item.ids
    pad_len = max_len - len(ids)

    if pad_len > 0:
        pad_ids = torch.tensor(ids + [pad_token_id] * pad_len, dtype=torch.long)
        attn_mask = torch.tensor([1] * len(ids) + [0] * pad_len, dtype=torch.long)
        
    else:
        pad_ids = torch.tensor(ids , dtype=torch.long)
        attn_mask = torch.tensor([1] * max_len, dtype=torch.long)

    input_list.append(pad_ids)
    mask_list.append(attn_mask)

input_tensor = torch.vstack(input_list).to(device)
attn_mask_tensor = torch.vstack(mask_list).to(device)

In [6]:
input_tensor 

tensor([[32000,  1036,   202,  1684,   377,   261,  5856,  7475,    17, 32001,
           224,   202, 32000,  3164,   202, 30370,   482,   261, 11202, 32001,
           224,   202,     1,     1],
        [32000,  1036,   202,  1684,   377,   261,  5856,  7475,    17, 32001,
           224,   202, 32000,  3164,   202, 18617,   261, 14079, 13253,  3486,
          3054, 32001,   224,   202]], device='cuda:1')

In [7]:
attn_mask_tensor

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:1')

In [8]:
# no kv-cache + atten_mask

max_output = 500

with torch.no_grad():

    
    for _ in range(max_output):

        logits = model.forward(input_tensor, attention_mask=attn_mask_tensor)
        # logits = model.forward(input_tensor, attention_mask=None)

        logits = logits[:, -1, :]

        # 概率采样
        probs = torch.softmax(
            logits,
            dim=-1
        )
        next_token = torch.multinomial(
            probs,
            num_samples=1
        )
        
        input_tensor = torch.cat([input_tensor, next_token], dim=1)
        attn_mask_tensor = torch.cat([attn_mask_tensor, torch.ones((attn_mask_tensor.shape[0], 1), device=attn_mask_tensor.device, dtype=attn_mask_tensor.dtype)], dim=1)

outputs = input_tensor.cpu().numpy()

for output in outputs:
    print(tokenizer.decode(output))
    print("----------------------------")

 system
You are a helpful assistant. 
 user
tell me a joke 
 Animals is an Asian diversion story that explores the shocking conclusion of Australian geneticist and IMPRO masterminds. Experience the thrill of accurate reporting, but learn from experiences and film producers that both seem to emanate from the same story.

Here's a breakdown of the story:

**The A* H**

Superman (1973) didn't die a single day but he did escape with his family, eventually blending his disability with newfound beliefs and roles in early adult films. This time, he was an avid golfer, recovering from the thrill of a riverboat. In the end, Superman kicked off his canoe journey with a return from a clowning in the early 40s.

Over the next 24 hours, Superman's friends, Rowan, Ryan, Jo, Ross and Will, embarked on a trip to Eleanor the Wise where they ride a glacial slide on a different road. A hilarious story featuring the legendary Geisha stand-off, also known as the "Henry Onkelos" saga.

The first person to r

In [10]:
# inference with attention mask

prompts = [
    apply_chat_template("tell me a joke"),
    apply_chat_template("write a python sort function")
]

pad_token_id = tokenizer.token_to_id("<pad>")

input_ids = tokenizer.encode_batch(prompts, add_special_tokens=False)

max_len = max(len(ids) for ids in input_ids)

input_list = []
mask_list = []

for item in input_ids:
    ids = item.ids
    pad_len = max_len - len(ids)

    if pad_len > 0:
        pad_ids = torch.tensor(ids + [pad_token_id] * pad_len, dtype=torch.long)
        attn_mask = torch.tensor([1] * len(ids) + [0] * pad_len, dtype=torch.long)
        
    else:
        pad_ids = torch.tensor(ids , dtype=torch.long)
        attn_mask = torch.tensor([1] * max_len, dtype=torch.long)

    input_list.append(pad_ids)
    mask_list.append(attn_mask)

input_tensor = torch.vstack(input_list).to(device)
attn_mask_tensor = torch.vstack(mask_list).to(device)

In [11]:
input_tensor

tensor([[32000,  1036,   202,  1684,   377,   261,  5856,  7475,    17, 32001,
           224,   202, 32000,  3164,   202, 30370,   482,   261, 11202, 32001,
           224,   202,     1,     1],
        [32000,  1036,   202,  1684,   377,   261,  5856,  7475,    17, 32001,
           224,   202, 32000,  3164,   202, 18617,   261, 14079, 13253,  3486,
          3054, 32001,   224,   202]], device='cuda:1')

In [12]:
# kv-cache + atten_mask

max_output = 200


with torch.no_grad():


    # 1. prefill
    logits, past_key_values = model.forward(input_tensor, use_cache=True, attention_mask=attn_mask_tensor)

    generated = input_ids


    for _ in range(max_output):

        logits = logits[:, -1, :]


        # 概率
        probs = torch.softmax(
            logits,
            dim=-1
        )
        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        input_tensor = torch.cat([input_tensor, next_token], dim=1)
        attn_mask_tensor = torch.cat([attn_mask_tensor, torch.ones((attn_mask_tensor.shape[0], 1), device=attn_mask_tensor.device, dtype=attn_mask_tensor.dtype)], dim=1)
        # NOTE: 使用kvcache时，只传入new token
        logits, past_key_values = model(next_token, past_key_values=past_key_values, use_cache=True, attention_mask=attn_mask_tensor)

outputs = input_tensor.cpu().numpy()

for output in outputs:
    print(tokenizer.decode(output))
    print("----------------------------")

 system
You are a helpful assistant. 
 user
tell me a joke 
-"•'I will never buy it"

I have always liked to joke around trading and sticking my arms out of his/her front sight. However, after reading the headlines, and having already earned my trust, everyone agreed to pay me a fair amount of money for selling it. I find that trading for it would be a disgrace. This is based on my contractual agreement with Andy Warhol. Zeeb, who played a key role in his creation and also played a part in playing a pivotal role in the photographic masterpiece, was a J.R.R. Patterson Colt in his last real-life spinoff. Brian Williams and Ed Blake, both fourth generation tuba and former backup skaters, helped develop a viable amplifier system that allowed for powerful bass outputs that were never encoded. I also enjoyed watching Terry Bradshaw, their current artist, discuss the importance of providing the opportunity for ownership to reach their potential. Fortunately, the situation took a turn
--------